# Annotated Landslide TIFF Crops and Session PNG Archive

This Stage 3 exporter reads `landslide.csv`. For each annotated landslide, it first checks whether `landslide/incident_{incident}_candidate_{candidate}_after.tif` already exists on Hugging Face. Existing crops are skipped before any raw TIFF download or crop work.

For each new entry, it fetches the Planet `incident_{incident}_after.tif` when present; if absent, it falls back to `incident_{incident}_gee_after.tif` in the same Hugging Face incident folder. It crops the annotation bounds and stages the cropped multi-band TIFF plus a square RGB PNG. The cropped TIFF is uploaded individually to `landslide/`; all PNGs generated in the run are accumulated into one `landslide_png/landslide_png_yyyymmdd_hhmmss.zip`.

The TIFFs and session ZIP are committed together, so a failed upload can be retried without leaving TIFFs that would cause the duplicate check to skip their PNGs. This notebook requires the Kaggle `huggingface_token` secret and an attached annotations dataset containing `landslide.csv`. It does not call the Planet API or process non-landslide annotations.

In [ ]:
import math
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, CommitOperationAdd, hf_hub_download

# ---------------- configuration ----------------
HF_REPO = "sasudo2/landslides"
HF_REPO_TYPE = "dataset"
HF_REVISION = "main"
HF_RAW_ROOT = "raw_images/raw_incidents"
HF_TIF_ROOT = "landslide"
HF_ZIP_ROOT = "landslide_png"

# Attach the annotation CSV dataset and adjust this path to its Kaggle mount point.
LANDSLIDE_CSV = Path("/kaggle/input/datasets/sanjayashrestha123/landslide-annotation/landslide.csv")

RAW_CACHE_DIR = Path("/kaggle/working/raw_after_cache")
TIFF_OUTPUT_DIR = Path("/kaggle/working/landslide")
PNG_OUTPUT_DIR = Path("/kaggle/working/landslide_png/session")
ZIP_OUTPUT_DIR = Path("/kaggle/working/landslide_png")
STRETCH_LO_PCT = 2.0
STRETCH_HI_PCT = 98.0
PNG_BRIGHTNESS_GAMMA = 1.0

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("huggingface_token")
api = HfApi(token=HF_TOKEN)
print(f"Hugging Face dataset: {HF_REPO}@{HF_REVISION}")

In [ ]:
# Load landslide annotations and skip existing crops before resolving or downloading raw images.
if not LANDSLIDE_CSV.is_file():
    raise FileNotFoundError(
        f"Missing landslide annotations at {LANDSLIDE_CSV}. Attach the CSV dataset or update LANDSLIDE_CSV."
    )
annotations = pd.read_csv(LANDSLIDE_CSV)
required = {"incident", "candidate", "label", "min_lon", "max_lon", "min_lat", "max_lat"}
missing = required - set(annotations.columns)
if missing:
    raise ValueError(f"{LANDSLIDE_CSV} is missing required columns: {sorted(missing)}")

annotations["label"] = annotations["label"].astype(str).str.strip().str.lower()
labels = set(annotations["label"].dropna())
if labels != {"landslide"}:
    raise ValueError(f"Expected only 'landslide' labels in {LANDSLIDE_CSV}, found {sorted(labels)}")
for column in ("incident", "candidate", "min_lon", "max_lon", "min_lat", "max_lat"):
    annotations[column] = pd.to_numeric(annotations[column], errors="coerce")
    if annotations[column].isna().any() or not np.isfinite(annotations[column]).all():
        raise ValueError(f"{LANDSLIDE_CSV} contains invalid values in {column!r}")
for column in ("incident", "candidate"):
    if (annotations[column] % 1 != 0).any():
        raise ValueError(f"{LANDSLIDE_CSV} contains non-integer {column} identifiers")
    annotations[column] = annotations[column].astype("int64").astype(str)
if (annotations["min_lon"] >= annotations["max_lon"]).any() or (annotations["min_lat"] >= annotations["max_lat"]).any():
    raise ValueError(f"{LANDSLIDE_CSV} contains invalid bounding boxes")
if annotations.duplicated(["incident", "candidate"]).any():
    raise ValueError("landslide.csv contains duplicate incident/candidate rows")

repo_files = set(api.list_repo_files(HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION))
tif_pattern = re.compile(rf"^{re.escape(HF_TIF_ROOT)}/incident_(\d+)_candidate_(\d+)_after\.tif$")
existing_tifs = {remote_path for remote_path in repo_files if tif_pattern.fullmatch(remote_path)}
raw_pattern = re.compile(
    rf"^{re.escape(HF_RAW_ROOT)}/incident_(\d+)/incident_\1_(after|gee_after)\.tif$"
)
raw_after_by_incident = {}
for remote_path in repo_files:
    match = raw_pattern.fullmatch(remote_path)
    if not match:
        continue
    incident, image_kind = match.groups()
    # Prefer Planet imagery when both sources are available.
    if incident not in raw_after_by_incident or image_kind == "after":
        raw_after_by_incident[incident] = {
            "path": remote_path,
            "source": "planet" if image_kind == "after" else "gee",
        }

work = []
missing_sources = []
skipped_existing = 0
for row in annotations.to_dict(orient="records"):
    stem = f"incident_{row['incident']}_candidate_{row['candidate']}_after"
    remote_tif = f"{HF_TIF_ROOT}/{stem}.tif"
    if remote_tif in existing_tifs:
        skipped_existing += 1
        continue
    raw_source = raw_after_by_incident.get(row["incident"])
    if raw_source is None:
        missing_sources.append(row["incident"])
        print(f"SKIP incident {row['incident']} candidate {row['candidate']}: neither Planet after nor GEE after TIFF is available")
        continue
    work.append({
        **row,
        "raw_path": raw_source["path"],
        "image_source": raw_source["source"],
        "remote_tif": remote_tif,
        "local_tif": TIFF_OUTPUT_DIR / f"{stem}.tif",
        "local_png": PNG_OUTPUT_DIR / f"{stem}.png",
    })
print(f"Landslide annotations: {len(annotations)}")
print(f"Existing cropped TIFFs skipped before raw download: {skipped_existing}")
print(f"Landslides skipped because both after-images are missing: {len(missing_sources)}")
print(f"New crops to process: {len(work)}")
print(f"Planet sources: {sum(row['image_source'] == 'planet' for row in work)}; GEE fallback sources: {sum(row['image_source'] == 'gee' for row in work)}")

In [ ]:
def crop_raw_after_to_tif(source_path, output_path, row):
    with rasterio.open(source_path) as src:
        if src.crs is None:
            raise ValueError(f"Raw after-image has no CRS: {source_path}")
        west, south, east, north = transform_bounds(
            "EPSG:4326", src.crs,
            float(row["min_lon"]), float(row["min_lat"]),
            float(row["max_lon"]), float(row["max_lat"]),
            densify_pts=21,
        )
        requested = from_bounds(west, south, east, north, src.transform)
        col0 = math.floor(requested.col_off + 1e-8)
        row0 = math.floor(requested.row_off + 1e-8)
        col1 = math.ceil(requested.col_off + requested.width - 1e-8)
        row1 = math.ceil(requested.row_off + requested.height - 1e-8)
        crop_width = col1 - col0
        crop_height = row1 - row0
        if crop_width <= 0 or crop_height <= 0:
            raise ValueError(f"Annotated box produced an empty crop for {source_path}")
        if crop_width > src.width or crop_height > src.height:
            raise ValueError(
                f"Annotated box for incident {row['incident']} candidate {row['candidate']} "
                "is larger than its raw after-image and cannot be shifted inside without resizing"
            )
        shifted_col0 = min(max(col0, 0), src.width - crop_width)
        shifted_row0 = min(max(row0, 0), src.height - crop_height)
        if (shifted_col0, shifted_row0) != (col0, row0):
            print(
                f"SHIFT incident {row['incident']} candidate {row['candidate']}: "
                f"crop origin ({col0}, {row0}) -> ({shifted_col0}, {shifted_row0}); "
                f"preserved {crop_width} x {crop_height} pixels"
            )

        window = Window(shifted_col0, shifted_row0, crop_width, crop_height)
        data = src.read(window=window)
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            width=data.shape[2],
            height=data.shape[1],
            transform=src.window_transform(window),
            tiled=False,
            compress="deflate",
        )
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(data)
            dst.update_tags(**src.tags())
            for band in range(1, src.count + 1):
                if src.descriptions[band - 1]:
                    dst.set_band_description(band, src.descriptions[band - 1])
                band_tags = src.tags(band)
                if band_tags:
                    dst.update_tags(band, **band_tags)
    return output_path


def download_raw_after(remote_path):
    return hf_hub_download(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        revision=HF_REVISION,
        filename=remote_path,
        token=HF_TOKEN,
        cache_dir=str(RAW_CACHE_DIR),
    )

In [ ]:
def write_rgb_png(tif_path, png_path):
    with rasterio.open(tif_path) as src:
        if src.count < 3:
            raise ValueError(f"Expected at least 3 bands in cropped Planet TIFF, found {src.count}: {tif_path}")
        # Planet RGB imagery is B,G,R[,N]; export standard RGB order as R,G,B.
        bands = src.read(indexes=[1, 2, 3], masked=True).astype(np.float32).filled(np.nan)
        preserve_uint8 = all(dtype == "uint8" for dtype in src.dtypes[:3])
    rgb = np.moveaxis(bands[[2, 1, 0]], 0, -1)
    if rgb.shape[0] == 0 or rgb.shape[1] == 0:
        raise ValueError(f"Cropped TIFF contains no pixels: {tif_path}")

    output = np.zeros(rgb.shape, dtype=np.uint8)
    for channel_index in range(3):
        channel = rgb[:, :, channel_index]
        valid = channel[np.isfinite(channel)]
        if valid.size == 0:
            raise ValueError(f"RGB crop has no valid pixels in channel {channel_index}")
        if preserve_uint8:
            scaled = np.clip(np.nan_to_num(channel, nan=0.0), 0, 255)
        else:
            low, high = np.percentile(valid, [STRETCH_LO_PCT, STRETCH_HI_PCT])
            if high <= low:
                scaled = np.zeros(channel.shape, dtype=np.float32)
            else:
                scaled = np.clip((channel - low) * (255.0 / (high - low)), 0, 255)
                scaled = np.nan_to_num(scaled, nan=0.0)
        if PNG_BRIGHTNESS_GAMMA != 1.0:
            scaled = 255.0 * (scaled / 255.0) ** PNG_BRIGHTNESS_GAMMA
        output[:, :, channel_index] = np.clip(scaled, 0, 255).astype(np.uint8)

    height, width, _ = output.shape
    if height != width:
        square = np.zeros((max(height, width), max(height, width), 3), dtype=np.uint8)
        square[:height, :width] = output
        output = square

    png_path = Path(png_path)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        png_path, "w", driver="PNG", width=output.shape[1], height=output.shape[0],
        count=3, dtype="uint8",
    ) as dst:
        dst.write(np.moveaxis(output, -1, 0))
    return png_path

In [ ]:
# Download only uncropped raw incidents, then stage the cropped TIFF and its PNG.
generated = []
processing_errors = []
for index, row in enumerate(work, start=1):
    try:
        raw_tif = download_raw_after(row["raw_path"])
        cropped_tif = crop_raw_after_to_tif(raw_tif, row["local_tif"], row)
        png_path = write_rgb_png(cropped_tif, row["local_png"])
        generated.append({**row, "local_tif": cropped_tif, "local_png": png_path})
        print(f"[{index}/{len(work)}] staged {row['remote_tif']} from {row['image_source'].upper()} and PNG")
    except Exception as exc:
        detail = f"incident {row['incident']} candidate {row['candidate']}: {type(exc).__name__}: {exc}"
        processing_errors.append(detail)
        print(f"ERROR {detail}")

print(f"Successful TIFF/PNG pairs: {len(generated)}; failed: {len(processing_errors)}")

In [ ]:
# Accumulate this session's PNGs in one ZIP. Commit all TIFFs and the ZIP together.
if generated:
    ZIP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run_tag = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    zip_path = ZIP_OUTPUT_DIR / f"landslide_png_{run_tag}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for row in generated:
            archive.write(row["local_png"], arcname=Path(row["local_png"]).name)

    operations = [
        CommitOperationAdd(path_in_repo=row["remote_tif"], path_or_fileobj=str(row["local_tif"]))
        for row in generated
    ]
    remote_zip = f"{HF_ZIP_ROOT}/{zip_path.name}"
    operations.append(CommitOperationAdd(path_in_repo=remote_zip, path_or_fileobj=str(zip_path)))
    api.create_commit(
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        revision=HF_REVISION,
        operations=operations,
        commit_message=f"Upload {len(generated)} annotated landslide crops and PNG archive",
    )
    print(f"Uploaded {len(generated)} individual cropped TIFFs under {HF_TIF_ROOT}/")
    print(f"Uploaded session PNG archive: {remote_zip}")
else:
    print("No new annotated landslides; no TIFFs or PNG archive to upload.")

In [ ]:
print(f"Cropped TIFF staging directory: {TIFF_OUTPUT_DIR}")
print(f"PNG staging directory: {PNG_OUTPUT_DIR}")
print(f"Existing cropped TIFFs skipped: {skipped_existing}")
print(f"Landslides skipped because both after-images are missing: {len(missing_sources)}")
if processing_errors:
    raise RuntimeError(
        f"Processing failed for {len(processing_errors)} annotated landslide(s). "
        "Successful TIFF/PNG pairs were uploaded; see the ERROR lines above and rerun after fixing the failures."
    )